# Preparação dos dados do INMET

Este notebook lê os arquivos horários das estações meteorológicas do INMET e produz uma tabela com uma linha por dia para o Sul do país. O caminho tem duas etapas: primeiro cada estação é agregada de horário para diário, e depois as estações são resumidas em um valor regional por dia, já que o ONS é analisado no nível do subsistema e não de uma cidade. Os arquivos originais ficam em data/raw/INMET, com uma pasta por ano e um CSV por estação.

In [9]:
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAW = Path("../data/raw/INMET")
INTERIM = Path("../data/interim")
INTERIM.mkdir(parents=True, exist_ok=True)

ANOS = range(2021, 2026)
INICIO = "2021-01-01"
FIM = "2025-12-31"

UTC_PARA_BRASILIA = pd.Timedelta(hours=-3)
MIN_HORAS_VALIDAS = 18
COBERTURA_MINIMA = 0.80

### Escolha das estações

O nome de cada arquivo já traz a região e o estado, como em INMET_S_PR_A807_CURITIBA. Os arquivos que começam com INMET_S são as estações do Paraná, Santa Catarina e Rio Grande do Sul, que formam o nosso recorte, então basta listá-los sem abrir as quase seiscentas estações do país. O estado e o código da estação também saem do nome do arquivo.

In [10]:
# Lista os arquivos de estações da região Sul nos anos pedidos.
def listar_arquivos_sul(anos):
    arquivos = []
    for ano in anos:
        arquivos += sorted((RAW / str(ano)).glob("INMET_S_*.CSV"))
    return arquivos


arquivos = listar_arquivos_sul(ANOS)
print(len(arquivos), "arquivos de estação")

494 arquivos de estação


### Leitura de um arquivo

Os arquivos do INMET têm algumas particularidades que a função de leitura precisa resolver. As oito primeiras linhas são metadados da estação, então a tabela só começa na nona. O separador é ponto e vírgula, o decimal é vírgula e a codificação é latin-1. Valores ausentes aparecem como -9999, que, se não fosse tratado, entraria nas médias como uma temperatura absurda. Os nomes das colunas têm acentos e unidades, então em vez de depender do nome exato localizamos cada variável pelo início do nome, sem acento, o que resiste melhor a pequenas diferenças entre anos.

Ficamos com três variáveis: chuva, temperatura e rajada de vento, que são as ligadas às tempestades que motivam a pergunta de pesquisa. As demais colunas, como pressão, umidade e radiação, ficam de fora para manter a base enxuta.

O ponto mais fácil de passar despercebido é o fuso. As horas do arquivo estão em UTC, e o dia local de Brasília começa às 03h UTC. Se agregássemos pela data UTC, a chuva da noite cairia no dia seguinte, e o ONS registra seus eventos no horário local. Por isso convertemos para UTC-3 antes de extrair a data. O Brasil não tem horário de verão desde 2019, então um deslocamento fixo basta para o nosso período.

Valores fora de uma faixa plausível viram ausentes, com limites folgados para não descartar fenômenos reais: chuva acima de 200 mm em uma hora, temperatura fora de -10 a 45 graus e rajada acima de 60 m/s. Cada passo guarda quantos valores foram descartados, para montarmos a tabela de qualidade mais adiante.

In [11]:
VARIAVEIS = {
    "chuva": "PRECIPITACAO TOTAL",
    "temperatura": "TEMPERATURA DO AR",
    "rajada": "VENTO, RAJADA",
}
LIMITES = {"chuva": (0, 200), "temperatura": (-10, 45), "rajada": (0, 60)}


# Remove acentos e padroniza caixa para comparar nomes de colunas.
def normalizar(texto):
    sem_acento = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return sem_acento.upper().strip()


# Lê um arquivo horário e devolve a tabela limpa em UTC e local, mais as contagens de problemas.
def ler_arquivo(caminho):
    tabela = pd.read_csv(caminho, sep=";", skiprows=8, decimal=",", encoding="latin-1")
    colunas = {normalizar(nome): nome for nome in tabela.columns}
    contagens = {"linhas": len(tabela)}

    data = pd.to_datetime(tabela.iloc[:, 0].astype(str).str.replace("/", "-"), format="%Y-%m-%d", errors="coerce")
    hora = pd.to_numeric(tabela.iloc[:, 1].astype(str).str[:2], errors="coerce")
    horario_utc = data + pd.to_timedelta(hora, unit="h")

    horario = pd.DataFrame({"horario_utc": horario_utc})
    horario["data"] = (horario_utc + UTC_PARA_BRASILIA).dt.normalize()

    for variavel, prefixo in VARIAVEIS.items():
        nome = next((original for normal, original in colunas.items() if normal.startswith(prefixo)), None)
        if nome is None:
            raise ValueError(f"{caminho.name}: coluna de {variavel} não encontrada")

        valores = pd.to_numeric(tabela[nome], errors="coerce").replace(-9999, np.nan)
        minimo, maximo = LIMITES[variavel]
        fora_da_faixa = (valores < minimo) | (valores > maximo)

        contagens[f"{variavel}_ausente"] = int(valores.isna().sum())
        contagens[f"{variavel}_fora_da_faixa"] = int(fora_da_faixa.sum())
        horario[variavel] = valores.where(~fora_da_faixa)

    contagens["datas_invalidas"] = int(horario["horario_utc"].isna().sum())
    horario = horario.dropna(subset=["horario_utc"])
    contagens["horarios_repetidos"] = int(horario.duplicated("horario_utc").sum())
    return horario.drop_duplicates("horario_utc"), contagens

### Agregação por dia em cada estação

Cada arquivo é agregado para diário logo depois de lido, o que mantém a memória baixa. A chuva do dia é a soma das horas, a temperatura máxima e a mínima vêm dos extremos horários, e a rajada do dia é o máximo, já que vento forte é um pico e não uma média. A temperatura horária subestima um pouco os extremos reais, mas a temperatura não é a variável central do problema, e isso evita lidar com as colunas de máxima e mínima da hora anterior.

Um dia só vale se tiver pelo menos 18 horas válidas da variável. Com menos que isso, uma soma de chuva subestima o acumulado real, e é preferível um valor ausente a um número enganoso. Pelo mesmo motivo a soma usa min_count: um dia inteiro sem leituras fica ausente em vez de virar 0 mm de chuva.

In [12]:
# Resume as leituras horárias de uma estação em um registro por dia.
def agregar_por_dia(horario):
    grupos = horario.groupby("data")
    diario = pd.DataFrame({
        "chuva": grupos["chuva"].sum(min_count=1),
        "temp_max": grupos["temperatura"].max(),
        "temp_min": grupos["temperatura"].min(),
        "rajada_max": grupos["rajada"].max(),
    })
    horas_validas = grupos[list(VARIAVEIS)].count()

    origem = {"chuva": "chuva", "temp_max": "temperatura", "temp_min": "temperatura", "rajada_max": "rajada"}
    for coluna, variavel in origem.items():
        diario[coluna] = diario[coluna].where(horas_validas[variavel] >= MIN_HORAS_VALIDAS)
    return diario.reset_index()


diarios, contagens = [], []
for caminho in arquivos:
    horario, contagem = ler_arquivo(caminho)
    _, _, uf, codigo, *_ = caminho.stem.split("_")

    diario = agregar_por_dia(horario)
    diario["uf"], diario["estacao"] = uf, codigo
    diarios.append(diario)
    contagens.append(contagem)

diario_estacoes = pd.concat(diarios, ignore_index=True)
diario_estacoes = diario_estacoes[diario_estacoes["data"].between(INICIO, FIM)]
print(len(diario_estacoes), "registros estação-dia")

175065 registros estação-dia


### Qualidade dos dados horários

Com tudo lido, a tabela abaixo mostra quanto de cada variável estava ausente nos arquivos originais e quanto passa a estar ausente depois de tratarmos os valores fora da faixa. Ela alimenta a parte do relatório que pede a documentação dos problemas encontrados, junto com as contagens de datas inválidas e horários repetidos.

In [13]:
total = pd.DataFrame(contagens).sum()

qualidade = pd.DataFrame({
    "% ausente original": [100 * total[f"{v}_ausente"] / total["linhas"] for v in VARIAVEIS],
    "% ausente após tratamento": [
        100 * (total[f"{v}_ausente"] + total[f"{v}_fora_da_faixa"]) / total["linhas"] for v in VARIAVEIS
    ],
}, index=list(VARIAVEIS)).round(2)

display(qualidade)
print("datas inválidas:", total["datas_invalidas"], "| horários repetidos removidos:", total["horarios_repetidos"])

,% ausente original,% ausente após tratamento
chuva,23.08,23.08
temperatura,19.71,19.71
rajada,24.94,24.94


datas inválidas: 0 | horários repetidos removidos: 0


### Seleção das estações

Agora decidimos quais estações entram. A rede do INMET cresce com os anos, então as estações mais novas só têm dados em parte do período. Se todas entrassem, o número de estações mudaria ao longo do tempo, e o máximo entre estações, que usaremos para capturar tempestades localizadas, tenderia a subir só porque há mais estações para produzir um valor extremo. Mantemos apenas as estações com pelo menos 80% dos dias válidos de chuva e temperatura, o que dá uma rede estável e comparável entre os anos. O histograma mostra como a cobertura se distribui e onde o corte cai.

In [ ]:
calendario = pd.date_range(INICIO, FIM, freq="D", name="data")

dia_valido = diario_estacoes["chuva"].notna() & diario_estacoes["temp_max"].notna()
cobertura = dia_valido.groupby([diario_estacoes["uf"], diario_estacoes["estacao"]]).sum() / len(calendario)

estacoes = cobertura.rename("cobertura").reset_index()
estacoes["selecionada"] = estacoes["cobertura"] >= COBERTURA_MINIMA

print(estacoes.groupby("uf")["selecionada"].agg(total="size", selecionadas="sum"))

### Resumo entre estações

Para cada dia combinamos as estações selecionadas em poucos valores regionais. A média de chuva descreve a região como um todo, mas dilui uma tempestade que atinge só uma parte dela, então guardamos também o máximo entre estações, que é o que sinaliza chuva forte localizada. Para a temperatura usamos médias, e para o vento apenas o máximo da rajada. O Rio Grande do Sul tem mais estações que os outros dois estados, então a média simples pesa mais o RS. Aceitamos isso por enquanto, e dá para revisar se a análise por estado se mostrar necessária. Também registramos quantas estações contribuíram em cada dia, como controle de qualidade.

In [15]:
selecionadas = diario_estacoes.merge(estacoes.loc[estacoes["selecionada"], ["uf", "estacao"]])

clima = selecionadas.groupby("data").agg(
    n_estacoes=("estacao", "nunique"),
    chuva_media=("chuva", "mean"),
    chuva_max=("chuva", "max"),
    temp_max_media=("temp_max", "mean"),
    temp_min_media=("temp_min", "mean"),
    rajada_max=("rajada_max", "max"),
).reindex(calendario)

clima["n_estacoes"] = clima["n_estacoes"].fillna(0).astype(int)
print("dias sem nenhuma estação:", (clima["n_estacoes"] == 0).sum())

dias sem nenhuma estação: 0


### Lacunas e exportação

Se algum dia ficou sem nenhuma estação válida, preenchemos com o último valor conhecido por até dois dias. Usamos o valor do passado e não uma interpolação linear porque interpolar usaria o dia seguinte para preencher o anterior, o que contradiz o horizonte de 24 horas do problema. Lacunas maiores permanecem ausentes. Salvamos a base regional e a tabela de estações, que será útil no relatório para dizer quais entraram em cada estado.

In [17]:
colunas_clima = clima.columns.drop("n_estacoes")
clima[colunas_clima] = clima[colunas_clima].ffill(limit=2)
print("valores ainda ausentes:\n", clima.isna().sum()[lambda x: x > 0])

clima.reset_index().to_parquet(INTERIM / "inmet_diario_sul.parquet", index=False)
estacoes.to_parquet(INTERIM / "inmet_estacoes_sul.parquet", index=False)

valores ainda ausentes:
 Series([], dtype: int64)


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.